# 1. Image denoising to reduce noise and enhance image quality

## This part of the code our deep learning approach (pre-trained DnCNN model) to denoise the image.

In [1]:
# Step 1: Mount Google Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
from glob import glob
from tqdm import tqdm
import pandas as pd
import torch
from torchvision.transforms import ToTensor

# Step 2: Set paths
original_folder = "../EMDS-6/EMDS5-Original"
output_folder = "EMDS5-Denoised-DnCnn"
device = "cuda" # if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Step 3: DnCNN Model Definition
class DnCNN(torch.nn.Module):
    def __init__(self, depth=17, n_channels=64, image_channels=1):
        super(DnCNN, self).__init__()
        layers = [
            torch.nn.Conv2d(image_channels, n_channels, kernel_size=3, padding=1),
            torch.nn.ReLU(inplace=True)
        ]
        for _ in range(depth - 2):
            layers += [
                torch.nn.Conv2d(n_channels, n_channels, kernel_size=3, padding=1),
                torch.nn.BatchNorm2d(n_channels),
                torch.nn.ReLU(inplace=True)
            ]
        layers += [torch.nn.Conv2d(n_channels, image_channels, kernel_size=3, padding=1)]
        self.dncnn = torch.nn.Sequential(*layers)

    def forward(self, x):
        noise = self.dncnn(x)
        return x - noise

# Step 4: Load Trained Model
model = DnCNN()
model.to(device)
model.eval()

# Step 5: Denoise and Evaluate
results = []

class_folders = sorted(os.listdir(original_folder))

for class_name in tqdm(class_folders, desc="Processing Classes"):
    class_input_path = os.path.join(original_folder, class_name)
    class_output_path = os.path.join(output_folder, class_name)
    os.makedirs(class_output_path, exist_ok=True)

    image_paths = sorted(glob(os.path.join(class_input_path, "*.png")))

    for img_path in image_paths:
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        img_norm = img.astype(np.float32) / 255.0
        tensor = ToTensor()(img_norm).unsqueeze(0).to(device)

        with torch.no_grad():
            denoised = model(tensor).squeeze().cpu().numpy()

        denoised_img = np.clip(denoised * 255.0, 0, 255).astype(np.uint8)

        filename = os.path.basename(img_path)
        save_path = os.path.join(class_output_path, filename)
        cv2.imwrite(save_path, denoised_img)

        # Compute Similarity (A) and Mean-Variance (S)
        N = img.size
        A = 1 - np.sum(np.abs(img.astype(np.float32) - denoised_img.astype(np.float32))) / (N * 255)
        S = 1 - np.sum((img.astype(np.float32) - denoised_img.astype(np.float32)) ** 2) / np.sum(img.astype(np.float32) ** 2)
        results.append((class_name, filename, round(A, 4), round(S, 4)))

# Step 6: Output summary table
df = pd.DataFrame(results, columns=["Class", "Filename", "Similarity_A", "MeanVariance_S"])
df_grouped = df.groupby("Class")[["Similarity_A", "MeanVariance_S"]].mean().reset_index()
print(df_grouped)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


Processing Classes: 100%|██████████| 21/21 [12:34<00:00, 35.95s/it]

   Class  Similarity_A  MeanVariance_S
0     01      0.992200        0.999777
1     02      0.992300        0.999850
2     03      0.992200        0.999805
3     04      0.992200        0.999788
4     05      0.992200        0.999765
5     06      0.992212        0.999790
6     07      0.992200        0.999845
7     08      0.992200        0.999795
8     09      0.992260        0.999873
9     10      0.992200        0.999830
10    11      0.992212        0.999472
11    12      0.992200        0.999765
12    13      0.992200        0.999772
13    14      0.992200        0.999855
14    15      0.992263        0.999673
15    16      0.992200        0.999845
16    17      0.992200        0.999770
17    18      0.992533        0.999833
18    19      0.992200        0.999835
19    20      0.992200        0.999817
20    21      0.992498        0.999810
